In [ ]:
# 導入必要的套件
import paho.mqtt.client as mqtt
import time
import json


In [ ]:
# MQTT Broker 設定（使用本地樹莓派上的 MQTT Broker）
BROKER = "localhost"  # 或使用 "127.0.0.1"
PORT = 1883
TOPIC = "工廠/溫度"  # 訂閱的主題（與 lesson1.ipynb 中的主題相同）

# 建立 MQTT 客戶端（使用最新版 API）
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)

# 連線回調函數
def on_connect(client, userdata, flags, rc, properties=None):
    if rc == 0:
        print(f"✅ 成功連接到 MQTT Broker: {BROKER}")
        # 連線成功後訂閱主題
        client.subscribe(TOPIC, qos=1)
        print(f"✅ 已訂閱主題: {TOPIC}")
    else:
        print(f"❌ 連線失敗，錯誤代碼: {rc}")

# 訊息接收回調函數
def on_message(client, userdata, msg):
    topic = msg.topic
    payload = msg.payload.decode('utf-8')
    print(f"\n📨 收到訊息:")
    print(f"   主題: {topic}")
    print(f"   內容: {payload}")
    
    # 嘗試解析為 JSON
    try:
        data = json.loads(payload)
        print(f"   JSON 解析成功:")
        for key, value in data.items():
            print(f"     {key}: {value}")
    except json.JSONDecodeError:
        print(f"   (純文字訊息)")

# 訂閱回調函數
def on_subscribe(client, userdata, mid, granted_qos, properties=None):
    print(f"✅ 訂閱確認，QoS: {granted_qos}")

# 設定回調函數
client.on_connect = on_connect
client.on_message = on_message
client.on_subscribe = on_subscribe

# 連接到 Broker
print(f"正在連接到 {BROKER}...")
client.connect(BROKER, PORT, 60)
client.loop_start()  # 開始背景執行緒處理訊息

# 等待連線建立
time.sleep(1)


In [ ]:
# 持續監聽訊息（等待 30 秒）
print("開始監聽訊息，等待 30 秒...")
print("(請在 lesson1.ipynb 中發送訊息來測試)")
print("-" * 50)

for i in range(30):
    time.sleep(1)
    if (i + 1) % 5 == 0:
        print(f"⏰ 已等待 {i+1} 秒...")

print("\n✅ 監聽時間結束")


In [ ]:
# 關閉連線
client.loop_stop()
client.disconnect()
print("✅ MQTT 連線已關閉")
